# 03 Train Symbolic SODT

- `F(x)`: the frozen detector up to RoI Align pooled RoI grids.
- `M(z)`: the original RoI classifier head that maps pooled RoIs to class logits.
- `z`: the pooled RoI feature grid itself, not the post-MLP `roi_embeddings`.
- The sparse oblique tree is trained offline in teacher-student mode on frozen RoIs, following the Hada/Kairgeldin split between neural feature extraction and symbolic classification.

Assumptions used here:
- the symbolic classifier includes the detector background class so it can replace the RoI classifier cleanly at inference time
- TAO decision-node reduced problems use `scikit-learn` L1 logistic regression with the `liblinear` solver, matching the solver family named in the Hada/Kairgeldin papers


In [ ]:
from pathlib import Path

import pandas as pd

from neuro.datasets import DeepPCBDataset
from neuro.inference import load_checkpoint_model
from neuro.transforms import build_eval_transforms
from neuro.utils import latest_run_checkpoint, load_yaml
from symbolic.export import export_teacher_roi_dataset
from symbolic.train import train_symbolic_tree_regularization_path
from symbolic.utils import load_symbolic_payload

PROJECT_ROOT = Path.cwd().resolve()


In [ ]:
model_config = load_yaml(PROJECT_ROOT / "configs/model.yaml")
train_config = load_yaml(PROJECT_ROOT / "configs/train_deeppcb.yaml")

dataset_root = PROJECT_ROOT / train_config["dataset"]["root"]
checkpoint_dir = PROJECT_ROOT / train_config["artifacts"]["checkpoint_dir"]
checkpoint_path = latest_run_checkpoint(checkpoint_dir)

symbolic_dir = PROJECT_ROOT / "checkpoints" / "symbolic"
teacher_export_path = symbolic_dir / "teacher_trainval.pt"
symbolic_checkpoint_path = symbolic_dir / "sodt_run1.pt"
symbolic_summary_path = symbolic_dir / "sodt_run1_summary.json"

checkpoint_path


In [ ]:
model, detector_checkpoint = load_checkpoint_model(
    checkpoint_path,
    PROJECT_ROOT / "configs/model.yaml",
    device=train_config.get("device"),
)

train_dataset = DeepPCBDataset(
    dataset_root=dataset_root,
    split_file=train_config["dataset"]["train_split"],
    transforms=build_eval_transforms(),
    class_names=tuple(model_config["model"]["class_names"]),
)

len(train_dataset), detector_checkpoint.get("metrics", {})


In [ ]:
export_teacher_roi_dataset(
    model=model,
    dataset=train_dataset,
    output_path=teacher_export_path,
    device=train_config.get("device"),
    max_positive_rois_per_image=24,
    max_background_rois_per_image=40,
    storage_dtype="float16",
)

teacher_export_path


In [ ]:
symbolic_manifest = load_symbolic_payload(teacher_export_path)
label_counts = pd.DataFrame(symbolic_manifest["records"])["label_counts"].apply(pd.Series).fillna(0).sum(axis=0)

pd.DataFrame(
    {
        "class_name": list(symbolic_manifest["class_names"]),
        "count": label_counts.reindex(range(len(symbolic_manifest["class_names"])), fill_value=0).astype(int).tolist(),
    }
)


In [ ]:
summary = train_symbolic_tree_regularization_path(
    export_path=teacher_export_path,
    output_path=symbolic_checkpoint_path,
    summary_path=symbolic_summary_path,
    max_depth=4,
    depth_values=[2, 3, 4],
    iterations=20,
    lambda_values=[1.0, 3.0, 10.0, 30.0],
    alpha_values=[0.0, 0.5, 1.0],
    logistic_max_iter=100,
    tolerance=1e-4,
    include_background=True,
    max_samples_total=4000,
    mimic_tolerance=0.005,
    macro_f1_tolerance=0.005,
    feature_screening={
        "enabled": True,
        "activation_threshold": 1e-8,
        "min_feature_std": 1e-5,
        "min_feature_range": 1e-5,
        "min_activation_rate": 0.0,
    },
    extended_thresholds={
        "min_mimic_accuracy": 0.98,
        "min_macro_f1_vs_teacher": 0.98,
        "min_necessity_advantage_over_random": 0.0,
        "min_sufficiency_advantage_over_random": 0.0,
    },
)

pd.DataFrame(summary["candidate_tables"]).sort_values(
    ["mimic_accuracy", "macro_f1_vs_teacher", "nonzero_weights", "tree_depth"],
    ascending=[False, False, True, True],
).reset_index(drop=True)


In [ ]:
interpretability_table = pd.DataFrame(
    summary["interpretability_review"]["comparison_table"]
)[
    [
        "candidate_id",
        "tree_depth",
        "mimic_accuracy",
        "macro_f1_vs_teacher",
        "active_internal_nodes",
        "mean_nonzero_per_node",
        "mean_path_feature_count",
        "box_grounded_roi_overlap",
        "pointing_score",
        "preserves_strong_faithfulness",
        "interpretability_gains",
        "recommended_intrinsic_order",
    ]
].sort_values(
    ["recommended_intrinsic_order", "tree_depth", "active_internal_nodes", "mean_nonzero_per_node"],
    ascending=[True, True, True, True],
).reset_index(drop=True)

display(interpretability_table)

{
    "feature_screening": summary["feature_screening"],
    "paper_faithful_selected": summary["selected_models"]["paper_faithful"],
    "thesis_extended_selected": summary["selected_models"]["thesis_extended"],
    "paper_faithful_selection": summary["selection"]["paper_faithful"],
    "thesis_extended_selection": summary["selection"]["thesis_extended"],
    "intrinsic_interpretability_review": summary["interpretability_review"],
}
